## Step 1: Set Up

In [4]:
# langchain-community chromadb pypdf python-dotenv# Install required packages
# %pip install -q langgraph langchain langchain-openai langchain-chroma 
# # pip install --upgrade --force-reinstall chromadb langchain-chroma opentelemetry-api opentelemetry-sdk             # In case chroma db doesn't work due to its new update
# %pip install -q 

In [ ]:
# Import Necessary Libraries

from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import PyPDF2
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from IPython.display import Image, display
from typing import Literal
import asyncio
from pathlib import Path
import os

print("Packages Successfully Imported")

Packages Successfully Imported


In [6]:
# Load API key
load_dotenv()
openai_api_key = os.getenv("api_key")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found! Please set it in your .env file.")

print("✅ API key loaded")

✅ API key loaded


In [7]:
# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=openai_api_key
)

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


## Step 2: Load PDF

In [28]:
document_path = r"C:\Users\Deborah Adelegan\Desktop\Working-With-RAG\files"
document_list = os.listdir(document_path)
pages = []

for file in document_list:
    # Full path construction
    file_path = os.path.join(document_path, file)
    
    # Check if file exists
    if not os.path.isfile(file_path):
        print(f"⚠️ File not found: {file_path}")
        print("Please update the file_path variable with your PDF file.")
        continue

        # print("\nFor this demo, we'll create sample documents instead...")
        # # Create sample documents for demo
        # from langchain_core.documents import Document
        # pages = [
        #     Document(page_content=" Drug design is part of the virtual component of artificial intelligence in which mathematical algorithms are used, which we discuss as machine learning",
        #             metadata={"page": 30}),
        #     Document(page_content="Artificial intelligence (AI) is a vast and exciting field with numerous potential advancements. These include enhanced neural networks, modular neural networks, explainable AI, evolving machine learning, and federallearning.",
        #             metadata={"page": 209}),
        #     Document(page_content="Clustering in biostatistics is an essential unsupervised machine learning technique.",
        #             metadata={"page": 322}),
        # ]
        # print("✅ Using sample documents for demo")
    else:        
        # Load the PDF
        loader = PyPDFLoader(file_path)
        pages = []
        
        # Load pages (async loading)
        async for page in loader.alazy_load():
            pages.append(page)
        
        print(f"✅ Loaded {len(pages)} pages from PDF")

Ignoring wrong pointing object 0 0 (offset 0)


✅ Loaded 468 pages from PDF
✅ Loaded 395 pages from PDF
✅ Loaded 538 pages from PDF
✅ Loaded 457 pages from PDF
✅ Loaded 391 pages from PDF
✅ Loaded 495 pages from PDF


Ignoring wrong pointing object 0 0 (offset 0)


✅ Loaded 524 pages from PDF
✅ Loaded 477 pages from PDF
✅ Loaded 146 pages from PDF


## Step 3: Split into Chunks

In [ ]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100)

# Split documents
doc_splits = text_splitter.split_documents(pages)

print(f"✅ Created {len(doc_splits)} chunks")
print(f"\nSample chunk:")
print(f"{doc_splits[56].page_content[:200]}...")

✅ Created 2240 chunks

Sample chunk:
for appropriate storage of data from a particular candidate in databases for
further application, which helps in identifying similar patterns in future. The
UK Biobank initiative in the United Kingdom...


## Step 4: Create Vector Store

In [ ]:
# Initialize embeddings using OpenAI

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key = openai_api_key
)

print("✅ Embeddings model initialized successfully")

✅ Embeddings model initialized successfully


In [ ]:
# Create chroma Vector store
chroma_path = 'chroma'

# Create vector from documents
vector_store = Chroma(
    collection_name="document",
    persist_directory=chroma_path,
    embedding_function=embeddings
)

# Add documents
vector_store.add_documents(documents=doc_splits)

print(f"✅ Vector store created with {len(doc_splits)} chunks")
print(f"   Persisted to: {chroma_path}")

✅ Vector store created with 2240 chunks
   Persisted to: chroma


#### Testing the Retrieval

In [ ]:
test_query = 'What is bionformatics?'
test_results = vector_store.similarity_search(test_query, k=3)

print(f"Query: {test_query}")
print(f"\nTop result:")
print(f"{test_results[0].page_content[:200]}...")
print(f"\n✅ Retrieval working!")

NameError: name 'vector_store' is not defined